# Train a Halloween postcard LoRA on FLUX 3 Image Klein

A LoRA is a small add-on that steers an image model toward a specific style. I trained one for FLUX 3 Image Klein on 65 Halloween postcards from the early 1900s. It adds the flat printed color and framed look I love in the originals.

## What you need

A Linux machine with a GPU that has at least 24 GB of VRAM, [AI Toolkit](https://github.com/ostris/ai-toolkit), and access to [FLUX 3 Image Klein base](https://huggingface.co/black-forest-labs/flux-3-image-klein-base) on Hugging Face. Get access before logging in. You'll also need Git, Python, and a CUDA-compatible PyTorch build for your driver.

The setup below puts the checkouts and outputs in a folder under your home directory. Change `work` if you want them somewhere else. For the released checkpoint, do the setup and then jump to **Use it**.

In [ ]:
from pathlib import Path
import os
import subprocess

work = Path.home() / "klein-halloween"
work.mkdir(parents=True, exist_ok=True)
toolkit = work / "ai-toolkit"
repo = work / "flux3-klein-lora-guide"
output = work / "output"
python = toolkit / "venv/bin/python"

In [ ]:
subprocess.run(["git", "clone", "https://github.com/ostris/ai-toolkit.git", str(toolkit)], check=True)
subprocess.run(["git", "checkout", "04733873cf1da8060fce01316ec45daedbd4a918"], cwd=toolkit, check=True)
subprocess.run(["python3", "-m", "venv", str(toolkit / "venv")], check=True)

I used PyTorch 2.13.0 for CUDA 13.0. Install a CUDA build that fits your driver first, then install AI Toolkit's requirements. The check should print `True`.

In [ ]:
subprocess.run([str(python), "-m", "pip", "install", "torch==2.13.0", "--index-url", "https://download.pytorch.org/whl/cu130"], check=True)
subprocess.run([str(python), "-m", "pip", "install", "-r", "requirements.txt"], cwd=toolkit, check=True)
subprocess.run([str(python), "-c", "import torch; print(torch.cuda.is_available())"], check=True)

Log in to Hugging Face with an account that has access to the model. If you can't type into the login prompt here, run `ai-toolkit/venv/bin/hf auth login` in a terminal from the work folder.

In [ ]:
!{toolkit}/venv/bin/hf auth login

## 1. The dataset

I used 65 Halloween postcards, some from [The Public Domain Review](https://publicdomainreview.org/collection/halloween-postcards/) and some from Wikimedia Commons. Keep a `metadata.csv` with each image's source and license, which is standard practice if you plan to share your LoRA.

Your images can be different sizes and shapes. I didn't crop any of mine. Look over your data before you train. With a small dataset every image counts.

In [ ]:
subprocess.run(["git", "clone", "https://github.com/MaanavD/flux3-klein-lora-guide.git", str(repo)], check=True)
images = sorted((repo / "halloween").glob("*.jpg"))
captions = sorted((repo / "halloween").glob("*.txt"))
print(f"{len(images)} images, {len(captions)} captions")
print("Images without captions:", [p.name for p in images if not p.with_suffix(".txt").exists()])

## 2. Captions and the trigger word

Each image has a `.txt` file with the same name in the same folder. Describe the people, objects, actions and setting, and any printed words. Leave the style out, because that's what the LoRA learns. Read the captions before you train.

Every caption starts with `vqx`. During training, anything the captions don't name ends up tied to that word, which here is the old postcard look. AI Toolkit can add a trigger with `trigger_word`, but these files already contain it, so the configs don't add it again.

I started with `pdhalloween`. The model could read "halloween" inside it, so it added Halloween things on its own. I switched to `vqx`. Before you train, put your trigger in a prompt on the base model. If the picture changes, pick another word.

In [ ]:
print((repo / "halloween/halloween_postcards_0001.txt").read_text())

## 3. The recipe

Most of AI Toolkit's defaults work great. I changed a few settings.

- Rank sets the size of the LoRA. I used rank 32, which is only 161 MB and about 80 million weights.
- I kept the learning rate at 1e-4 and tried five schedules. My favorites were `cosine` and `constant_with_warmup`, which warms up for 200 steps and then stays constant.
- `timestep_type: shift` practices on high noise about as often as Klein's sampler encounters it. In my runs, the same prompts looked like printed cards by step 300 with `shift`.
- I trained for 4,000 steps and saved every 100 to compare. My favorites were at 2,000, though some runs looked good at 4,000 too. A much bigger dataset may need longer.

The rest are AI Toolkit defaults or settings that keep memory down. I used AdamW, batch size 1, 1024 px, bf16, gradient checkpointing and latents cached to disk. The repo has both complete configs, including the sample prompts.

Choose `cosine` or `warmup`. This copies the selected config and fills in the two paths. The originals stay untouched.

In [ ]:
schedule = "cosine"  # change to "warmup" for constant_with_warmup
assert schedule in ("cosine", "warmup")
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "PyYAML"], check=True)
import yaml

config = yaml.safe_load((repo / f"configs/halloween-lora-{schedule}.yaml").read_text())
process = config["config"]["process"][0]
process["training_folder"] = str(output)
process["datasets"][0]["folder_path"] = str(repo / "halloween")
output.mkdir(parents=True, exist_ok=True)
config_path = work / f"halloween-lora-{schedule}.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())

## 4. Train and watch the samples

First, a 50-step smoke test. It saves and samples every 25 steps and should take no longer than 15 to 20 minutes. It has its own run name so its checkpoints stay separate from the full run.

In [ ]:
import copy
smoke = copy.deepcopy(config)
smoke["config"]["name"] += "-smoke"
smoke_process = smoke["config"]["process"][0]
smoke_process["train"]["steps"] = 50
smoke_process["save"]["save_every"] = 25
smoke_process["sample"]["sample_every"] = 25
smoke_path = work / f"halloween-lora-{schedule}-smoke.yaml"
smoke_path.write_text(yaml.safe_dump(smoke, sort_keys=False))
subprocess.run([str(python), "run.py", str(smoke_path)], cwd=toolkit, check=True)

If the smoke test works, start the full run. The preview prompts are your test questions. They cover a cat with and without `vqx`, scenes close to the cards, new subjects like a robot and mecha, and Christmas scenes with and without the trigger. The same seeds let you compare checkpoints fairly.

In [ ]:
subprocess.run([str(python), "run.py", str(config_path)], cwd=toolkit, check=True)

Open the checkpoints and `samples/` in the run folder and judge them by eye. In my cosine run, the witch prompt already looked like a printed card at step 300, and settled into flat printed color by steps 1,000 and 2,000. The subjects your cards never show are the better test.

In [ ]:
run_dir = output / config["config"]["name"]
print("Run folder:", run_dir)
print("Samples:", run_dir / "samples")
print("Checkpoints:", *sorted(run_dir.glob("*.safetensors")), sep="\n")

## 5. Pick a checkpoint

Compare the same prompts and seeds across checkpoints. Past some point a checkpoint can learn the training images too well, copying their details instead of just the style. I compared five schedules at steps 2,000 and 4,000 with the base model. My favorites were cosine and constant_with_warmup, both at 2,000 steps.

A cosine schedule depends on how long the whole run is. My pick came from step 2,000 of a 4,000-step run, where the learning rate was still at half its peak. Ending a run at 2,000 would bring the rate to zero there and could look different.

## 6. Use it

Woohoo, we made it! Pick a checkpoint from your run, or download my released cosine checkpoint below. You can also change `cosine` to `warmup` to download the other favorite. Run one of these two cells.

In [ ]:
# Use your own step 2,000 checkpoint, or change the path to your favorite.
checkpoint = run_dir / f"{config['config']['name']}_000002000.safetensors"
assert checkpoint.is_file(), checkpoint
print(checkpoint)

In [ ]:
# Skip training: download a released step 2,000 checkpoint.
from urllib.request import urlretrieve
release_schedule = "cosine"  # or "warmup"
assert release_schedule in ("cosine", "warmup")
filename = f"klein-vqx-shift4k-{release_schedule}_000002000.safetensors"
checkpoint = work / filename
urlretrieve(f"https://github.com/MaanavD/flux3-klein-lora-guide/releases/download/checkpoints-2000/{filename}", checkpoint)
print(checkpoint)

The script loads the checkpoint on Klein. Put `vqx` in the prompt and describe what you want on the card. I used strength 1. You can change the strength to set how strongly the LoRA shows, so experiment with it.

In [ ]:
prompt = """vqx, an old Halloween greeting card with a hand-drawn illustration of a scarecrow standing in a pumpkin patch at dusk under a large harvest moon, with an owl on the fence post, black crows on the scarecrow's arms and corn shocks behind him. The card has the clearly readable greeting "Best Wishes for Halloween" across the top."""
env = os.environ.copy()
env["PYTHONPATH"] = str(toolkit) + os.pathsep + env.get("PYTHONPATH", "")
card = work / "card.jpg"
subprocess.run([str(python), str(repo / "scripts/make_card.py"), "--lora", str(checkpoint),
                "--prompt", prompt, "--out", str(card), "--strength", "1"],
               cwd=toolkit, env=env, check=True)
print(card)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(card)))

## Make your own

Now swap in images you care about. Gather them, record their sources and licenses, caption what's pictured, pick a trigger word, train, and compare checkpoints. The Public Domain Review and Wikimedia Commons are good places to find images you can use. Have fun with your own datasets, and use them responsibly.

Make some spooky cards and send them to people. Have a happy Halloween!